In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import pytz
import re

df = pd.read_csv('googleplaystore.csv')

df['Installs'] = df['Installs'].astype(str).str.replace(r'[+,]', '', regex=True)
df['Installs'] = pd.to_numeric(df['Installs'], errors='coerce')

df['Price'] = df['Price'].astype(str).str.replace('$', '', regex=False)
df['Price'] = pd.to_numeric(df['Price'], errors='coerce').fillna(0)
df['Revenue'] = df['Installs'] * df['Price']

def parse_size(size_str):
    if pd.isna(size_str) or size_str == 'Varies with device':
        return np.nan
    if 'M' in size_str:
        return float(size_str.replace('M', ''))
    elif 'k' in size_str:
        return float(size_str.replace('k', '')) / 1024
    return np.nan

df['Size_MB'] = df['Size'].apply(parse_size)

df['Android_Ver_Num'] = df['Android Ver'].astype(str).str.extract(r'^(\d+\.\d+)').astype(float)

df['App_Name_Len'] = df['App'].astype(str).apply(len)

filtered_df = df[
    (df['Installs'] >= 10000) & 
    (df['Size_MB'] > 15.0) & 
    (df['Content Rating'] == 'Everyone') & 
    (df['App_Name_Len'] <= 30) & 
    (df['Android_Ver_Num'] > 4.0) &
    ((df['Revenue'] >= 10000) | (df['Type'] == 'Free'))
]

top_3_categories = filtered_df.groupby('Category')['Installs'].sum().nlargest(3).index
top_df = filtered_df[filtered_df['Category'].isin(top_3_categories)]

final_data = top_df.groupby(['Category', 'Type']).agg({
    'Installs': 'mean',
    'Revenue': 'mean'
}).unstack(fill_value=0)

ist = pytz.timezone('Asia/Kolkata')
current_time = datetime.now(ist).time()
start_time = datetime.strptime("13:00", "%H:%M").time()
end_time = datetime.strptime("14:00", "%H:%M").time()

if start_time <= current_time <= end_time:
    categories = final_data.index
    x = np.arange(len(categories))
    width = 0.2
    
    fig, ax1 = plt.subplots(figsize=(12, 6))
    
    bar1 = ax1.bar(x - width, final_data['Installs']['Free'], width, label='Free - Avg Installs', color='#3498db')
    bar2 = ax1.bar(x, final_data['Installs']['Paid'], width, label='Paid - Avg Installs', color='#2980b9')
    
    ax1.set_xlabel('App Category', fontweight='bold')
    ax1.set_ylabel('Average Installs', color='#2980b9', fontweight='bold')
    ax1.tick_params(axis='y', labelcolor='#2980b9')
    ax1.set_xticks(x)
    ax1.set_xticklabels(categories)
    
    ax2 = ax1.twinx()
    bar3 = ax2.bar(x + width, final_data['Revenue']['Paid'], width, label='Paid - Avg Revenue ($)', color='#2ecc71')
    
    ax2.set_ylabel('Average Revenue ($)', color='#27ae60', fontweight='bold')
    ax2.tick_params(axis='y', labelcolor='#27ae60')
    
    fig.legend(loc='upper right', bbox_to_anchor=(0.9, 0.9))
    plt.title('Top 3 Categories: Avg Installs vs. Revenue (Free vs Paid)', fontweight='bold')
    
    fig.tight_layout()
    plt.show()
else:
    print(f"Chart hidden. Current IST Time: {current_time.strftime('%I:%M %p')}.")

Chart hidden. Current IST Time: 07:05 PM.
